# ForecastPipeline: End-to-End Automation

In production forecasting, reproducibility and consistency are critical. Manual workflows —
loading data, fitting models, generating forecasts, evaluating accuracy — are error-prone
and hard to maintain as the number of targets and models grows.

**ForecastPipeline** solves this by orchestrating the full forecasting cycle in a single,
configurable object:

1. **Data** — load and validate the data source
2. **Preprocess** — apply transformations (log, diff, outlier removal, etc.)
3. **Fit** — estimate all configured models
4. **Forecast** — generate point forecasts and prediction intervals
5. **Combine** — optionally combine forecasts (mean, median, BMA, OLS)
6. **Evaluate** — compute out-of-sample metrics via cross-validation
7. **Report** — produce a structured summary

This notebook demonstrates the complete pipeline lifecycle, including **RecurringForecast**
for periodic re-estimation and pipeline serialization.

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecastbox.pipeline import ForecastPipeline, RecurringForecast, PipelineStep

# Add helpers path
sys.path.insert(0, "../utils")
from helpers import (
    load_macro_brazil,
    create_pipeline_config,
    save_pipeline_config,
    load_pipeline_config,
)

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("forecastbox pipeline modules loaded successfully.")

## 1. Pipeline Configuration

A `ForecastPipeline` is configured with:
- **data_source** — a DataFrame, Series, or callable that returns data
- **target** — the column to forecast
- **models** — list of model names to fit (e.g., `auto_arima`, `auto_ets`, `naive`)
- **combination** — how to combine forecasts (`mean`, `median`, `bma`, `ols`, or `None`)
- **evaluation** — metrics to compute (`rmse`, `mae`, `mape`)
- **horizon** — number of periods ahead to forecast
- **preprocess** — preprocessing steps (`log`, `diff`, `detrend`, `standardize`, `outlier_detection`, `missing_fill`)

We also use a helper function to create and save a JSON configuration for documentation.

In [ ]:
# Load the Brazilian macroeconomic dataset
df = load_macro_brazil()
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df.index[0]} to {df.index[-1]}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
display(df.head())

# Create a configuration dict for documentation purposes
config = create_pipeline_config(
    target="ipca",
    models=["auto_arima", "auto_ets", "naive"],
    horizon=12,
    retrain_frequency="monthly",
)
print("\nPipeline configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

# Build the ForecastPipeline
pipeline = ForecastPipeline(
    data_source=df,
    target="ipca",
    models=["auto_arima", "auto_ets", "naive"],
    combination="mean",
    evaluation=["rmse", "mae"],
    horizon=12,
    preprocess=["missing_fill"],
)

print(f"\nPipeline created: {pipeline}")
print(f"Steps: {pipeline.steps()}")

## 2. Data Preparation Step

The first pipeline step resolves the data source and applies preprocessing
transformations. We can run individual steps with `run_step()` to inspect
intermediate results.

Available preprocessing steps:
- `missing_fill` — linear interpolation for NaN values
- `outlier_detection` — IQR-based outlier replacement
- `log` — log transformation
- `diff` — first difference
- `seasonal_diff` — seasonal difference (period=12)
- `detrend` — remove linear trend
- `standardize` — zero mean, unit variance

In [ ]:
# Step 1: Data — resolve data source and extract target column
raw_data = pipeline.run_step("data")
print(f"Target series: '{raw_data.name}', length={len(raw_data)}")
print(f"Date range: {raw_data.index[0]} to {raw_data.index[-1]}")
print(f"Missing values: {raw_data.isna().sum()}")

# Step 2: Preprocess — apply configured transformations
processed = pipeline.run_step("preprocess")
print(f"\nAfter preprocessing: length={len(processed)}, NaN={processed.isna().sum()}")

# Visualize raw vs processed
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(raw_data.index, raw_data.values, "b-", linewidth=1.2)
axes[0].set_title("Raw IPCA (Monthly Inflation)", fontsize=12)
axes[0].set_ylabel("% change")
axes[0].grid(True, alpha=0.3)

axes[1].plot(processed.index, processed.values, "g-", linewidth=1.2)
axes[1].set_title("After Preprocessing (missing_fill)", fontsize=12)
axes[1].set_ylabel("% change")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBasic statistics:")
display(processed.describe())

## 3. Model Training Step

The `fit` step estimates all configured models on the preprocessed data.
Each model receives the full training series and stores fitted parameters
for later forecasting.

In [ ]:
# Step 3: Fit — train all models
fitted_models = pipeline.run_step("fit")

print(f"Models trained: {len(fitted_models)}")
print()

for name, info in fitted_models.items():
    print(f"Model: {name}")
    print(f"  Data length: {info['data_length']}")
    print(f"  Fitted mean: {info['mean']:.4f}")
    print(f"  Fitted std:  {info['std']:.4f}")
    print(f"  Trend:       {info['trend']:.6f}")
    print()

## 4. Forecasting Step

Generate point forecasts and prediction intervals for the configured horizon.
When a combination method is set, the pipeline also produces a combined forecast.

In [ ]:
# Step 4: Forecast — generate predictions for horizon=12
forecasts = pipeline.run_step("forecast")

print(f"Forecasts generated for {len(forecasts)} models:\n")

fig, axes = plt.subplots(1, len(forecasts), figsize=(5 * len(forecasts), 5))
if len(forecasts) == 1:
    axes = [axes]

for ax, (name, fc) in zip(axes, forecasts.items()):
    fc_df = fc.to_dataframe()
    display(fc_df.head())
    
    # Plot historical + forecast
    ax.plot(processed.index[-60:], processed.values[-60:], "k-", linewidth=1.2, label="Historical")
    ax.plot(fc.index, fc.point, "b-", linewidth=2, label="Forecast")
    if fc.lower_95 is not None:
        ax.fill_between(fc.index, fc.lower_95, fc.upper_95, alpha=0.15, color="blue", label="95% CI")
    if fc.lower_80 is not None:
        ax.fill_between(fc.index, fc.lower_80, fc.upper_80, alpha=0.25, color="blue", label="80% CI")
    ax.set_title(f"{name}", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Individual Model Forecasts (h=12)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Also show the combined forecast
combined = pipeline.run_step("combine")
if combined is not None:
    print(f"\nCombined forecast ({combined.model_name}):")
    print(f"  Point forecast (first 6): {combined.point[:6].round(4)}")
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(processed.index[-60:], processed.values[-60:], "k-", linewidth=1.2, label="Historical")
    for name, fc in forecasts.items():
        ax.plot(fc.index, fc.point, "--", alpha=0.5, label=name)
    ax.plot(combined.index, combined.point, "r-", linewidth=2.5, label=f"Combined ({combined.model_name})")
    if combined.lower_95 is not None:
        ax.fill_between(combined.index, combined.lower_95, combined.upper_95, alpha=0.1, color="red")
    ax.set_title("Individual vs Combined Forecasts", fontsize=14, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Evaluation Step

The `evaluate` step computes out-of-sample metrics for each model using cross-validation.
Metrics include RMSE, MAE, and any other configured measures.

In [ ]:
# Step 6: Evaluate — compute metrics for each model
eval_df = pipeline.run_step("evaluate")

print("Evaluation Metrics by Model:")
print("=" * 40)
display(eval_df.style.highlight_min(axis=0, color="lightgreen"))

# Visualize metrics comparison
fig, axes = plt.subplots(1, len(eval_df.columns), figsize=(6 * len(eval_df.columns), 5))
if len(eval_df.columns) == 1:
    axes = [axes]

colors = ["steelblue", "darkorange", "forestgreen"]
for ax, col in zip(axes, eval_df.columns):
    bars = ax.bar(eval_df.index, eval_df[col], color=colors[:len(eval_df)])
    ax.set_title(col.upper(), fontsize=14, fontweight="bold")
    ax.set_ylabel(col.upper())
    ax.grid(True, alpha=0.3, axis="y")
    # Annotate values
    for bar, val in zip(bars, eval_df[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{val:.4f}", ha="center", va="bottom", fontsize=10)

plt.suptitle("Model Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Full Pipeline Run

Instead of running steps individually, `pipeline.run()` executes the entire DAG
in one call and returns a `PipelineResults` object with forecasts, evaluation,
cross-validation results, and execution timing.

In [ ]:
# Run the complete pipeline in one call
results = pipeline.run()

# Print the full summary report
print(results.summary())

# Inspect the results object
print(f"\nForecasts: {list(results.forecasts.keys())}")
print(f"Combination: {results.combination.model_name if results.combination else 'None'}")
print(f"Best model: {results.best_model()}")
print(f"CV results: {list(results.cv_results.keys())}")

# Show evaluation table
print("\nEvaluation:")
display(results.evaluation)

# Serialize to dict for inspection
results_dict = results.to_dict()
print(f"\nSerialized keys: {list(results_dict.keys())}")
print(f"Metadata: {results.metadata}")

## 7. RecurringForecast

In practice, forecasts are updated periodically as new data arrives. `RecurringForecast`
wraps a pipeline and manages re-estimation history, enabling:

- **Forecast evolution tracking** — how predictions change over time
- **Revision analysis** — quantify revisions between consecutive runs
- **Visual diagnostics** — plot forecast evolution

We simulate 6 monthly re-estimations by adding one new observation at a time.

In [ ]:
# Simulate 6 months of recurring forecasts by progressively extending the data
# We hold out the last 6 observations and add them back one at a time

n_months = 6
base_end = len(df) - n_months

# Create a data updater that extends the data each time
month_counter = {"current": 0}

def data_updater():
    """Simulate new data arriving each month."""
    end_idx = base_end + month_counter["current"]
    month_counter["current"] += 1
    return df.iloc[:end_idx]

# Create pipeline with the data updater
recurring_pipeline = ForecastPipeline(
    data_source=df.iloc[:base_end],  # initial data
    target="ipca",
    models=["auto_arima", "auto_ets", "naive"],
    combination="mean",
    evaluation=["rmse", "mae"],
    horizon=12,
    preprocess=["missing_fill"],
)

# Set up RecurringForecast with monthly frequency
recurring = RecurringForecast(
    pipeline=recurring_pipeline,
    frequency="monthly",
    data_updater=data_updater,
)

print(f"RecurringForecast: {recurring}")
print(f"Simulating {n_months} monthly re-estimations...\n")

# Run 6 monthly executions
for i in range(n_months):
    result = recurring.run_once()
    best = result.best_model()
    total_time = sum(result.execution_time.values())
    print(f"  Month {i + 1}: best_model={best}, time={total_time:.3f}s")

print(f"\nTotal executions: {len(recurring.history())}")

# Forecast evolution: how predictions changed across executions
evolution = recurring.forecast_evolution()
print(f"\nForecast evolution (first 6 horizons):")
display(evolution.iloc[:, :6])

# Revision analysis
revisions = recurring.revision_analysis()
print("\nRevision analysis:")
display(revisions)

# Plot forecast evolution
fig, ax = plt.subplots(figsize=(14, 6))
recurring.plot_evolution(ax=ax)
ax.set_title("Forecast Evolution Across Monthly Re-estimations", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Saving and Loading Pipelines

Pipeline configurations can be saved to JSON for reproducibility and sharing.
The helper functions `save_pipeline_config` / `load_pipeline_config` handle
serialization, and the pipeline can be reconstructed from the saved config.

In [ ]:
import json
from pathlib import Path
import tempfile

# Save pipeline configuration to JSON
config = create_pipeline_config(
    target="ipca",
    models=["auto_arima", "auto_ets", "naive"],
    horizon=12,
    retrain_frequency="monthly",
)

# Add extra pipeline-specific settings
config["combination"] = "mean"
config["evaluation_metrics"] = ["rmse", "mae"]
config["preprocess"] = ["missing_fill"]
config["cv_type"] = "expanding"

# Save to a temporary file (in production, use a permanent path)
save_path = Path(tempfile.gettempdir()) / "my_pipeline.json"
save_pipeline_config(config, save_path)
print(f"Pipeline config saved to: {save_path}")
print(f"\nSaved config:")
print(json.dumps(config, indent=2))

# Load back and reconstruct the pipeline
loaded_config = load_pipeline_config(save_path)
print(f"\nLoaded config matches: {loaded_config == config}")

# Reconstruct pipeline from loaded config
loaded_pipeline = ForecastPipeline(
    data_source=df,
    target=loaded_config["target"],
    models=loaded_config["models"],
    combination=loaded_config.get("combination"),
    evaluation=loaded_config.get("evaluation_metrics", ["rmse"]),
    horizon=loaded_config["horizon"],
    preprocess=loaded_config.get("preprocess", []),
)
print(f"\nReconstructed pipeline: {loaded_pipeline}")

# Verify it produces the same results
loaded_results = loaded_pipeline.run()
print(f"Loaded pipeline best model: {loaded_results.best_model()}")
print(f"Original pipeline best model: {results.best_model()}")

# Clean up
save_path.unlink(missing_ok=True)

### Exercise 1: Build pipeline for inflation forecasting

Create a `ForecastPipeline` for IPCA inflation using 3 different models (`auto_arima`,
`auto_ets`, `naive`). Apply `outlier_detection` and `missing_fill` preprocessing.
Run the full pipeline with `combination='median'` and compare the combined forecast
to individual models.

In [ ]:
# TODO: Exercise 1 - Pipeline for inflation with 3 models
# Hints:
# 1. Load macro_brazil with load_macro_brazil()
# 2. Create ForecastPipeline with target="ipca", combination="median"
# 3. Add preprocess=["outlier_detection", "missing_fill"]
# 4. Run pipeline.run() and inspect results.summary()
# 5. Plot individual vs combined forecasts

### Exercise 2: Compare monthly vs quarterly re-estimation

Set up two `RecurringForecast` instances — one with `frequency='monthly'` and one with
`frequency='quarterly'`. Simulate 12 months of data arrival. Compare forecast revisions
and evolution between the two frequencies. Which one produces more stable forecasts?

In [ ]:
# TODO: Exercise 2
# Hints:
# 1. Create two ForecastPipeline instances (same config)
# 2. Wrap each with RecurringForecast(pipeline, frequency='monthly') and frequency='quarterly'
# 3. For monthly: run_once() 12 times with data_updater adding 1 month each time
# 4. For quarterly: run_once() 4 times with data_updater adding 3 months each time
# 5. Compare revision_analysis() and plot_evolution() for both
# 6. Discuss: monthly is more responsive, quarterly is more stable